In [ ]:
import ROOT

ROOT_FILE = "/data2/segmentlinking/CMSSW_12_5_0_pre3/RelValTTbar_14TeV_CMSSW_12_5_0_pre3/event_1000.root"
TREE_PATH = "trackingNtuple/tree"

OUTPUT_FILE = "tree_structure.txt"


In [ ]:
# ─────────────────────────────────────────────
# open file and tree
# ─────────────────────────────────────────────

f = ROOT.TFile.Open(ROOT_FILE)
t = f.Get(TREE_PATH)

print("Opened tree with", t.GetEntries(), "events")


# ─────────────────────────────────────────────
# save branch structure to file
# ─────────────────────────────────────────────

ROOT.gSystem.RedirectOutput(OUTPUT_FILE, "w")
t.Print()
ROOT.gSystem.RedirectOutput("", "")

print(f"Tree structure written to {OUTPUT_FILE}")


# ─────────────────────────────────────────────
# inspect first few events
# ─────────────────────────────────────────────

N_EVENTS = 5

print("\nInspecting first events:\n")

for i in range(min(N_EVENTS, t.GetEntries())):

    t.GetEntry(i)

    # example sim-track branches
    # adjust names after we inspect tree_structure.txt

    if hasattr(t, "sim_pt"):
        n = len(t.sim_pt)
        print(f"Event {i}: sim tracks = {n}")

        if n > 0:
            print("   first sim_pt:", t.sim_pt[0])

    if hasattr(t, "sim_eta"):
        print("   sim_eta size:", len(t.sim_eta))

    if hasattr(t, "sim_phi"):
        print("   sim_phi size:", len(t.sim_phi))

In [ ]:
#imports for the root file rip

from __future__ import annotations

import math
import os
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import ROOT

In [ ]:
# Paralell data rip, splits up into n-chunks then writes each to a npz file then we merge

# we implement some filters like the naive eta limitation etc

# ─────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────

ROOT_FILE = "/data2/segmentlinking/CMSSW_12_5_0_pre3/RelValTTbar_14TeV_CMSSW_12_5_0_pre3/event_1000.root"
TREE_PATH = "trackingNtuple/tree"

OUTDIR = Path("track_chunks_etaM1to1")
FINAL_OUTFILE = "track_data_etaM1to1.npz"

ETA_MIN = -1.0
ETA_MAX = 1.0
REQUIRE_NVALID_POSITIVE = True
REQUIRE_CHARGED = True

N_WORKERS = 20

FEATURE_BRANCHES = [
    "sim_q",
    "sim_pdgId",
    "sim_nValid",
    "sim_nPixel",
    "sim_nStrip",
    "sim_nLay",
    "sim_nPixelLay",
    "sim_n3DLay",
    "sim_nTrackerHits",
    "sim_nRecoClusters",
]

LABEL_BRANCHES = [
    "sim_pca_pt",
    "sim_pca_eta",
    "sim_pca_phi",
    "sim_pca_dxy",
    "sim_pca_dz",
]

# These are only for filtering / sanity checks
FILTER_BRANCHES = [
    "sim_q",
    "sim_nValid",
    "sim_pca_eta",
]


# ─────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────

def vec_to_np(v, dtype=np.float32):
    return np.asarray(list(v), dtype=dtype)


def all_same_length(named_arrays: dict[str, np.ndarray]) -> bool:
    if not named_arrays:
        return True
    lengths = [len(a) for a in named_arrays.values()]
    return len(set(lengths)) == 1


def split_ranges(n_events: int, n_chunks: int) -> list[tuple[int, int, int]]:
    ranges = []
    chunk_size = math.ceil(n_events / n_chunks)
    for chunk_id in range(n_chunks):
        start = chunk_id * chunk_size
        end = min((chunk_id + 1) * chunk_size, n_events)
        if start < end:
            ranges.append((chunk_id, start, end))
    return ranges


# ─────────────────────────────────────────────
# Worker
# ─────────────────────────────────────────────

def process_chunk(chunk_id: int, evt_start: int, evt_end: int, outdir_str: str):
    outdir = Path(outdir_str)
    outdir.mkdir(parents=True, exist_ok=True)

    f = ROOT.TFile.Open(ROOT_FILE)
    if not f or f.IsZombie():
        raise RuntimeError(f"[chunk {chunk_id}] Could not open ROOT file")

    t = f.Get(TREE_PATH)
    if not t:
        raise RuntimeError(f"[chunk {chunk_id}] Could not get tree {TREE_PATH}")

    feature_chunks = []
    label_chunks = []

    n_total_rows = 0
    n_kept_rows = 0
    n_bad_length_events = 0
    n_empty_events = 0

    for i_evt in range(evt_start, evt_end):
        t.GetEntry(i_evt)

        feat_dict = {
            name: vec_to_np(getattr(t, name), dtype=np.float32)
            for name in FEATURE_BRANCHES
        }
        lab_dict = {
            name: vec_to_np(getattr(t, name), dtype=np.float32)
            for name in LABEL_BRANCHES
        }
        filt_dict = {
            name: vec_to_np(getattr(t, name), dtype=np.float32)
            for name in FILTER_BRANCHES
        }

        all_dict = {}
        all_dict.update(feat_dict)
        all_dict.update(lab_dict)
        all_dict.update(filt_dict)

        if not all_same_length(all_dict):
            n_bad_length_events += 1
            continue

        n = len(next(iter(all_dict.values())))
        if n == 0:
            n_empty_events += 1
            continue

        n_total_rows += n

        X_evt = np.column_stack([feat_dict[name] for name in FEATURE_BRANCHES]).astype(np.float32, copy=False)
        Y_evt = np.column_stack([lab_dict[name] for name in LABEL_BRANCHES]).astype(np.float32, copy=False)

        sim_q = filt_dict["sim_q"]
        sim_nValid = filt_dict["sim_nValid"]
        sim_pca_eta = filt_dict["sim_pca_eta"]

        mask = np.ones(n, dtype=bool)

        # finite checks
        mask &= np.all(np.isfinite(X_evt), axis=1)
        mask &= np.all(np.isfinite(Y_evt), axis=1)
        mask &= np.isfinite(sim_q)
        mask &= np.isfinite(sim_nValid)
        mask &= np.isfinite(sim_pca_eta)

        # eta cut
        mask &= (sim_pca_eta >= ETA_MIN) & (sim_pca_eta <= ETA_MAX)

        # charged only
        if REQUIRE_CHARGED:
            mask &= (sim_q != 0)

        # nonzero/valid-hit requirement
        if REQUIRE_NVALID_POSITIVE:
            mask &= (sim_nValid > 0)

        X_evt = X_evt[mask]
        Y_evt = Y_evt[mask]

        if len(X_evt) > 0:
            feature_chunks.append(X_evt)
            label_chunks.append(Y_evt)
            n_kept_rows += len(X_evt)

    if feature_chunks:
        X = np.concatenate(feature_chunks, axis=0)
        Y = np.concatenate(label_chunks, axis=0)
    else:
        X = np.empty((0, len(FEATURE_BRANCHES)), dtype=np.float32)
        Y = np.empty((0, len(LABEL_BRANCHES)), dtype=np.float32)

    outfile = outdir / f"chunk_{chunk_id:03d}.npz"
    np.savez_compressed(
        outfile,
        X=X,
        Y=Y,
        evt_start=evt_start,
        evt_end=evt_end,
        n_total_rows=n_total_rows,
        n_kept_rows=n_kept_rows,
        n_bad_length_events=n_bad_length_events,
        n_empty_events=n_empty_events,
    )

    return {
        "chunk_id": chunk_id,
        "evt_start": evt_start,
        "evt_end": evt_end,
        "outfile": str(outfile),
        "shape_X": X.shape,
        "shape_Y": Y.shape,
        "n_total_rows": n_total_rows,
        "n_kept_rows": n_kept_rows,
        "n_bad_length_events": n_bad_length_events,
        "n_empty_events": n_empty_events,
    }


# ─────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────

def main():
    OUTDIR.mkdir(parents=True, exist_ok=True)

    f = ROOT.TFile.Open(ROOT_FILE)
    if not f or f.IsZombie():
        raise RuntimeError(f"Could not open ROOT file: {ROOT_FILE}")

    t = f.Get(TREE_PATH)
    if not t:
        raise RuntimeError(f"Could not get tree: {TREE_PATH}")

    n_events = t.GetEntries()
    print(f"Opened tree with {n_events} events")

    ranges = split_ranges(n_events, N_WORKERS)
    print(f"Using {len(ranges)} worker chunks")

    results = []
    with ProcessPoolExecutor(max_workers=len(ranges)) as ex:
        futures = [
            ex.submit(process_chunk, chunk_id, start, end, str(OUTDIR))
            for chunk_id, start, end in ranges
        ]

        for fut in as_completed(futures):
            res = fut.result()
            results.append(res)
            print(
                f"[done] chunk {res['chunk_id']:03d} "
                f"events {res['evt_start']}:{res['evt_end']} "
                f"X{res['shape_X']} Y{res['shape_Y']} "
                f"kept {res['n_kept_rows']:,}/{res['n_total_rows']:,}"
            )

    results.sort(key=lambda r: r["chunk_id"])

    X_parts = []
    Y_parts = []

    total_rows = 0
    kept_rows = 0
    bad_events = 0
    empty_events = 0

    for res in results:
        data = np.load(res["outfile"], allow_pickle=True)
        X_parts.append(data["X"])
        Y_parts.append(data["Y"])
        total_rows += int(data["n_total_rows"])
        kept_rows += int(data["n_kept_rows"])
        bad_events += int(data["n_bad_length_events"])
        empty_events += int(data["n_empty_events"])

    X = np.concatenate(X_parts, axis=0) if X_parts else np.empty((0, len(FEATURE_BRANCHES)), dtype=np.float32)
    Y = np.concatenate(Y_parts, axis=0) if Y_parts else np.empty((0, len(LABEL_BRANCHES)), dtype=np.float32)

    np.savez_compressed(
        FINAL_OUTFILE,
        X=X,
        Y=Y,
        feature_names=np.array(FEATURE_BRANCHES, dtype=object),
        label_names=np.array(LABEL_BRANCHES, dtype=object),
    )

    print("\nFinal merged dataset:")
    print("X shape:", X.shape)
    print("Y shape:", Y.shape)
    print(f"Total raw rows: {total_rows:,}")
    print(f"Total kept rows: {kept_rows:,}")
    print(f"Bad-length events skipped: {bad_events}")
    print(f"Empty events skipped: {empty_events}")
    print(f"Saved final dataset to {FINAL_OUTFILE}")


if __name__ == "__main__":
    main()

In [ ]:
from __future__ import annotations

import numpy as np

NPZ_FILE = "track_data_etaM1to1.npz"

data = np.load(NPZ_FILE, allow_pickle=True)

X = data["X"]
Y = data["Y"]
feature_names = [str(x) for x in data["feature_names"]]
label_names = [str(x) for x in data["label_names"]]

print("Loaded:", NPZ_FILE)
print("X shape:", X.shape)
print("Y shape:", Y.shape)
print()

# ----------------------------------------
# 1. Structural checks
# ----------------------------------------

assert X.ndim == 2, "X must be 2D"
assert Y.ndim == 2, "Y must be 2D"
assert X.shape[0] == Y.shape[0], "Row count mismatch between X and Y"
assert X.shape[1] == len(feature_names), "Feature count mismatch"
assert Y.shape[1] == len(label_names), "Label count mismatch"

print("Basic shape checks passed.")

x_finite = np.isfinite(X)
y_finite = np.isfinite(Y)

print("All X finite:", bool(np.all(x_finite)))
print("All Y finite:", bool(np.all(y_finite)))

# ----------------------------------------
# 2. Column lookup maps
# ----------------------------------------

feature_idx = {name: i for i, name in enumerate(feature_names)}
label_idx = {name: i for i, name in enumerate(label_names)}

print()
print("Feature names:", feature_names)
print("Label names:", label_names)

# ----------------------------------------
# 3. Filter compliance checks
# ----------------------------------------

sim_q = X[:, feature_idx["sim_q"]]
sim_nvalid = X[:, feature_idx["sim_nValid"]]
sim_pca_eta = Y[:, label_idx["sim_pca_eta"]]

print()
print("Charge filter checks:")
print("  min sim_q:", sim_q.min())
print("  max sim_q:", sim_q.max())
print("  all sim_q != 0:", bool(np.all(sim_q != 0)))

print()
print("Eta filter checks:")
print("  min sim_pca_eta:", sim_pca_eta.min())
print("  max sim_pca_eta:", sim_pca_eta.max())
print("  all sim_pca_eta in [-1,1]:", bool(np.all((sim_pca_eta >= -1.0) & (sim_pca_eta <= 1.0))))

print()
print("nValid filter checks:")
print("  min sim_nValid:", sim_nvalid.min())
print("  all sim_nValid > 0:", bool(np.all(sim_nvalid > 0)))

# ----------------------------------------
# 4. Column summaries
# ----------------------------------------

print()
print("Feature summaries:")
for i, name in enumerate(feature_names):
    col = X[:, i]
    print(
        f"{name:16s}  "
        f"min={col.min(): .6g}  "
        f"max={col.max(): .6g}  "
        f"mean={col.mean(): .6g}  "
        f"std={col.std(): .6g}"
    )

print()
print("Label summaries:")
for i, name in enumerate(label_names):
    col = Y[:, i]
    print(
        f"{name:16s}  "
        f"min={col.min(): .6g}  "
        f"max={col.max(): .6g}  "
        f"mean={col.mean(): .6g}  "
        f"std={col.std(): .6g}"
    )

# ----------------------------------------
# 5. All-zero column checks
# ----------------------------------------

print()
print("Checking for all-zero columns:")
for i, name in enumerate(feature_names):
    print(f"{name:16s} all zero? {bool(np.all(X[:, i] == 0))}")
for i, name in enumerate(label_names):
    print(f"{name:16s} all zero? {bool(np.all(Y[:, i] == 0))}")

# ----------------------------------------
# 6. Constant-column checks
# ----------------------------------------

print()
print("Checking for constant columns:")
for i, name in enumerate(feature_names):
    print(f"{name:16s} constant? {bool(np.all(X[:, i] == X[0, i]))}")
for i, name in enumerate(label_names):
    print(f"{name:16s} constant? {bool(np.all(Y[:, i] == Y[0, i]))}")

# ----------------------------------------
# 7. Duplicate-row spot checks
# ----------------------------------------

print()
print("Basic row sanity checks:")
print("  Any rows in X all zero?", bool(np.any(np.all(X == 0, axis=1))))
print("  Any rows in Y all zero?", bool(np.any(np.all(Y == 0, axis=1))))

# ----------------------------------------
# 8. Correlation checks that actually match this dataset
# ----------------------------------------

def corr(a, b):
    if np.std(a) == 0 or np.std(b) == 0:
        return np.nan
    return np.corrcoef(a, b)[0, 1]

print()
print("Some useful correlations in this pulled dataset:")

if "sim_q" in feature_idx and "sim_pca_pt" in label_idx:
    print(
        "corr(sim_q, sim_pca_pt):",
        corr(X[:, feature_idx["sim_q"]], Y[:, label_idx["sim_pca_pt"]])
    )

if "sim_nValid" in feature_idx and "sim_pca_pt" in label_idx:
    print(
        "corr(sim_nValid, sim_pca_pt):",
        corr(X[:, feature_idx["sim_nValid"]], Y[:, label_idx["sim_pca_pt"]])
    )

if "sim_nTrackerHits" in feature_idx and "sim_pca_pt" in label_idx:
    print(
        "corr(sim_nTrackerHits, sim_pca_pt):",
        corr(X[:, feature_idx["sim_nTrackerHits"]], Y[:, label_idx["sim_pca_pt"]])
    )

if "sim_nRecoClusters" in feature_idx and "sim_pca_pt" in label_idx:
    print(
        "corr(sim_nRecoClusters, sim_pca_pt):",
        corr(X[:, feature_idx["sim_nRecoClusters"]], Y[:, label_idx["sim_pca_pt"]])
    )

if "sim_pca_eta" in label_idx and "sim_pca_pt" in label_idx:
    print(
        "corr(sim_pca_eta, sim_pca_pt):",
        corr(Y[:, label_idx["sim_pca_eta"]], Y[:, label_idx["sim_pca_pt"]])
    )

# ----------------------------------------
# 9. Print a few sample paired rows
# ----------------------------------------

print()
print("Sample paired rows:")

n_samples = min(3, X.shape[0])
sample_rows = np.random.default_rng(0).choice(X.shape[0], size=n_samples, replace=False)

for r in sample_rows:
    print(f"\nRow {int(r)}")
    print("  Features:")
    for i, name in enumerate(feature_names):
        print(f"    {name:16s} = {X[r, i]: .6g}")
    print("  Labels:")
    for i, name in enumerate(label_names):
        print(f"    {name:16s} = {Y[r, i]: .6g}")

print()
print("Validation script finished.")

In [ ]:
# Unique lengths??

import ROOT 
import numpy as np

ROOT_FILE = "/data2/segmentlinking/CMSSW_12_5_0_pre3/RelValTTbar_14TeV_CMSSW_12_5_0_pre3/event_1000.root"
TREE_PATH = "trackingNtuple/tree"

BRANCHES = [
    "sim_q",
    "sim_pdgId",
    "sim_nValid",
    "sim_nPixel",
    "sim_nStrip",
    "sim_nLay",
    "sim_nPixelLay",
    "sim_n3DLay",
    "sim_nTrackerHits",
    "sim_nRecoClusters",
    "sim_pca_pt",
    "sim_pca_eta",
    "sim_pca_phi",
    "sim_pca_dxy",
    "sim_pca_dz",
]

def vec_to_np(v):
    return np.asarray(list(v), dtype=np.float32)

f = ROOT.TFile.Open(ROOT_FILE)
t = f.Get(TREE_PATH)

for i_evt in range(min(10, t.GetEntries())):
    t.GetEntry(i_evt)
    print(f"\nEvent {i_evt}")
    lengths = {}
    for name in BRANCHES:
        arr = vec_to_np(getattr(t, name))
        lengths[name] = len(arr)
        print(f"  {name:16s} len = {len(arr)}")
    unique_lengths = sorted(set(lengths.values()))
    print("  unique lengths:", unique_lengths)